# TSGC — Full Benchmark: Three Compression Architectures

**Paper:** *Temporal Semantic Gradient Compression for Cross-Platform LLM Session Portability*  
**Author:** Utkarsh Aggarwal · [github.com/Utkarsh-Aggarwal/UniMemo](https://github.com/Utkarsh-Aggarwal/UniMemo)

---

## Three Architectures

| Method | Inspired by | Retention decided by |
|---|---|---|
| **TSGC** | Fixed zones | Position (recency proxy) |
| **TSGC-AG** | LSTM gates | Position + content novelty |
| **TSGC-AT** | Transformer attention | Cross-message cosine similarity |

All three baselines also compared: RAW, TRUNCATE, UNIFORM, DEFLATE-ONLY

## Key Hypothesis
> For **linear** conversations (topic stays the same), recency is a good proxy for importance — TSGC works well.  
> For **non-linear** conversations (topic callbacks, reference to old messages), attention-based scoring outperforms recency — TSGC-AT wins.

In [ ]:
!pip install -q rouge-score matplotlib seaborn pandas numpy scikit-learn

In [ ]:
import re, zlib, json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter
from rouge_score import rouge_scorer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

plt.rcParams.update({
    'figure.facecolor':'#0d0f14','axes.facecolor':'#161921',
    'axes.edgecolor':'#2a2d3a','text.color':'#f0f2f8',
    'axes.labelcolor':'#8b90a8','xtick.color':'#8b90a8',
    'ytick.color':'#8b90a8','grid.color':'#1e2330',
    'grid.linestyle':'--','axes.titlecolor':'#f0f2f8',
    'font.family':'DejaVu Sans','axes.titlesize':13,'axes.labelsize':11,
})
ACCENT='#7c6fe0'; SUCCESS='#22c55e'; WARNING='#f59e0b'
DANGER='#ef4444'; MUTED='#4e5368'; BLUE='#4285F4'
TEAL='#06b6d4'; PINK='#ec4899'
print('Setup complete')

## 1. Dataset — Linear + Non-Linear Conversations

We include a **non-linear conversation** (Conv 4) where early messages are explicitly referenced later.  
This is the key scenario where TSGC-AT should outperform position-based methods.

In [ ]:
import json
import urllib.request
import os

# Try to load local dataset first, otherwise fallback to GitHub (for Colab)
try:
    with open('dataset.json', 'r') as f:
        CONVERSATIONS = json.load(f)
    print("Loaded local dataset.json")
except FileNotFoundError:
    print("Downloading dataset.json from GitHub...")
    url = "https://raw.githubusercontent.com/Utkarsh-Aggarwal/UniMemo/main/research/dataset.json"
    response = urllib.request.urlopen(url)
    CONVERSATIONS = json.loads(response.read().decode('utf-8'))
    print("Download complete.")

for name, conv in CONVERSATIONS.items():
    chars = sum(len(m['content']) for m in conv)
    print(f'{name:28} {len(conv):3} messages  {chars:8,} chars')


## 2. Core Utilities

In [ ]:
SIGNAL_WORDS = {
    'because','therefore','however','but','although','instead','error',
    'result','conclusion','solution','problem','issue','answer','key',
    'important','note','warning','step','must','should','never','always',
    'define','means','returns','function','class','approach','algorithm',
    'difference','summary','finally','specifically','example','critical',
    'insight','benefit','concern','pattern','principle','method','technique',
}

FILLER_RE = [
    r'^(sure|of course|certainly|absolutely|great|okay)[!.,]?\s*',
    r'^(as i (mentioned|said|noted|explained) (earlier|above|before))',
    r'^(to (summarize|recap|reiterate))[,:]?',
    r'hope (this|that) (helps|clarifies)',
    r'^(great question|good question)[!.]?',
]

def split_sentences(text):
    sents = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sents if len(s.strip()) > 10]

def is_filler(s):
    return any(re.search(p, s.strip(), re.IGNORECASE) for p in FILLER_RE)

def score_sentence(s, i, total):
    rel  = i / max(total-1, 1)
    pos  = 1 - abs(rel-0.5)*0.6
    words = re.findall(r'\b[a-z]{3,}\b', s.lower())
    dens = len(set(words))/len(words) if words else 0
    kw   = min(1.0, sum(1 for w in words if w in SIGNAL_WORDS)/3)
    return 0.3*pos + 0.4*dens + 0.3*kw

def extract_top(text, ratio, role='assistant'):
    sents = split_sentences(text)
    if len(sents)<=1: return text
    if role=='assistant':
        cleaned = [s for s in sents if not is_filler(s)] or sents
    else:
        cleaned = sents
    keep  = max(1, round(len(cleaned)*ratio))
    scored = sorted(enumerate(cleaned), key=lambda x: -score_sentence(x[1],x[0],len(cleaned)))
    top   = sorted(scored[:keep], key=lambda x: x[0])
    return ' '.join(s for _,s in top)

def effective_bytes(msgs, deflate=True):
    text = json.dumps([{'role':m['role'],'content':m['content']} for m in msgs])
    raw  = text.encode()
    return len(zlib.compress(raw, 6)) if deflate else len(raw)

scorer_rouge = rouge_scorer.RougeScorer(['rouge1','rougeL'], use_stemmer=True)

def key_term_retention(orig, comp):
    ot = ' '.join(m['content'] for m in orig).lower()
    ct = ' '.join(m['content'] for m in comp).lower()
    sigs = set(re.findall(r'\b[a-z]{3,}\b', ot)) & SIGNAL_WORDS
    if not sigs: return 1.0
    return sum(1 for w in sigs if w in ct)/len(sigs)

def recency_fidelity(orig, comp, n=3):
    ref   = ' '.join(m['content'] for m in orig[-n:])
    cands = [m for m in comp if m.get('_zone',1)==1] or comp[-n:]
    hyp   = ' '.join(m['content'] for m in cands[-n:])
    if not hyp.strip(): return 0.0
    return scorer_rouge.score(ref, hyp)['rougeL'].fmeasure

def info_density(msgs):
    ds=[]
    for m in msgs:
        w=re.findall(r'\b[a-z]{3,}\b',m['content'].lower())
        if w: ds.append(len(set(w))/len(w))
    return np.mean(ds) if ds else 0

def pivot_recall(orig, comp):
    """How well are the most-referenced (pivot) messages preserved?
    A pivot message is one whose key terms appear most often in later messages."""
    n = len(orig)
    ref_counts = [0]*n
    for i in range(n):
        words_i = set(re.findall(r'\b[a-z]{4,}\b', orig[i]['content'].lower()))
        for j in range(i+1, n):
            words_j = set(re.findall(r'\b[a-z]{4,}\b', orig[j]['content'].lower()))
            ref_counts[i] += len(words_i & words_j)
    if not any(ref_counts): return 1.0
    top_k = sorted(range(n), key=lambda x: -ref_counts[x])[:max(1,n//3)]
    pivot_text = ' '.join(orig[i]['content'] for i in top_k)
    comp_text  = ' '.join(m['content'] for m in comp)
    if not comp_text.strip(): return 0.0
    return scorer_rouge.score(pivot_text, comp_text)['rougeL'].fmeasure

def evaluate(name, orig, comp, deflate=True):
    ob = effective_bytes(orig, deflate=False)
    cb = effective_bytes(comp, deflate=deflate)
    return {
        'Method': name,
        'Savings %':        round((1-cb/ob)*100, 1),
        'Key Term %':       round(key_term_retention(orig,comp)*100, 1),
        'Recency %':        round(recency_fidelity(orig,comp)*100, 1),
        'Pivot Recall %':   round(pivot_recall(orig,comp)*100, 1),
        'Info Density %':   round(info_density(comp)*100, 1),
        'Msgs Kept':        len(comp),
    }

print('Utilities ready')

## 3. All Compression Methods
### 3a. Baselines

In [ ]:
def method_raw(msgs):            return msgs
def method_truncate(msgs, k=8): return msgs[-k:]
def method_uniform(msgs, r=0.5):
    return [{'role':m['role'],'content':extract_top(m['content'],r,m['role'])} for m in msgs]
def method_deflate_only(msgs):   return msgs  # deflate applied at eval time

print('Baselines ready')

### 3b. TSGC — Position-Based (Baseline TSGC)

In [ ]:
def method_tsgc(msgs, z1=5, z2=15):
    """Layer 1: dedup. Layer 2: fixed zone ratios by position. Layer 3: DEFLATE at eval."""
    # Layer 1: deduplication
    seen, deduped = set(), []
    for m in msgs:
        sents  = split_sentences(m['content'])
        unique = []
        for s in sents:
            k = s.strip().lower()
            if len(k)<20 or k not in seen:
                seen.add(k); unique.append(s)
        c = ' '.join(unique).strip()
        if c: deduped.append({'role':m['role'],'content':c})

    # Layer 2: temporal gradient by position
    result, n = [], len(deduped)
    for i, m in enumerate(deduped):
        ri = n-1-i  # 0=most recent
        if   ri < z1:  zone, ratio = 1, 1.0
        elif ri < z2:  zone, ratio = 2, 0.40
        else:          zone, ratio = 3, 0.15
        c = extract_top(m['content'], ratio, m['role']) if ratio<1.0 and len(m['content'])>80 else m['content']
        result.append({'role':m['role'],'content':c,'_zone':zone})
    return result

print('TSGC ready')

### 3c. TSGC-AG — Adaptive Gating (LSTM-Inspired)

Maintains running state between messages:  
- **Novelty gate**: if this message introduces many new signal words → protect it more  
- **Similarity gate**: if this message is very similar to the previous → compress it more  
- Final ratio = base_ratio × (1 + novelty_bonus) × (1 − similarity_penalty)

In [ ]:
def method_tsgc_ag(msgs, z1=5, z2=15):
    """TSGC with Adaptive Gating — LSTM-style novelty + similarity gates."""

    # Layer 1: deduplication (same as TSGC)
    seen, deduped = set(), []
    for m in msgs:
        sents  = split_sentences(m['content'])
        unique = [s for s in sents if len(s.strip())<20 or s.strip().lower() not in seen]
        for s in unique: seen.add(s.strip().lower())
        c = ' '.join(unique).strip()
        if c: deduped.append({'role':m['role'],'content':c})

    # Layer 2-AG: adaptive gating
    result, n = [], len(deduped)
    cell_state = set()   # accumulated signal words seen so far (long-term memory)
    prev_words = set()   # words from previous message (short-term)

    for i, m in enumerate(deduped):
        ri = n-1-i
        # Base ratio from position zone (same as TSGC)
        if   ri < z1:  zone, base_ratio = 1, 1.0
        elif ri < z2:  zone, base_ratio = 2, 0.40
        else:          zone, base_ratio = 3, 0.15

        curr_words = set(re.findall(r'\b[a-z]{4,}\b', m['content'].lower()))
        curr_signals = curr_words & SIGNAL_WORDS

        # ── Novelty gate (Input gate analogue) ──────────────────
        # How many signal words in this message haven't been seen before?
        new_signals = curr_signals - cell_state
        novelty = len(new_signals) / max(len(curr_signals), 1)
        novelty_bonus = novelty * 0.4   # up to +40% on ratio

        # ── Similarity gate (Forget gate analogue) ───────────────
        # How similar is this message to the previous one?
        overlap = len(curr_words & prev_words) / max(len(curr_words | prev_words), 1)
        sim_penalty = overlap * 0.3     # up to -30% on ratio

        # ── Adaptive ratio ───────────────────────────────────────
        adaptive_ratio = min(1.0, max(0.1, base_ratio + novelty_bonus - sim_penalty))

        # ── Update cell state (long-term memory) ─────────────────
        cell_state |= curr_signals
        prev_words  = curr_words

        c = extract_top(m['content'], adaptive_ratio, m['role']) \
            if adaptive_ratio < 0.95 and len(m['content']) > 80 else m['content']

        result.append({
            'role':m['role'],'content':c,'_zone':zone,
            '_novelty':round(novelty,2),'_overlap':round(overlap,2),
            '_ratio':round(adaptive_ratio,2)
        })
    return result

print('TSGC-AG ready')

### 3d. TSGC-AT — Attention-Based (Transformer-Inspired)

Computes **cross-message cosine similarity** (TF-IDF attention) to identify pivotal messages.  
A message that many others attend to (high centrality) → high importance → protected from compression.  
This is analogous to self-attention in Transformers, without any learned weights.

In [ ]:
def message_importance(msgs):
    """Compute importance score per message via TF-IDF cosine attention.
    
    Importance(i) = Σ_j sim(msg_i, msg_j) for j ≠ i
    
    High importance = many other messages are semantically related to this one.
    This identifies 'pivot' messages that anchor the conversation.
    """
    texts = [m['content'] for m in msgs]
    if len(texts) < 2:
        return [1.0] * len(texts)

    # TF-IDF vectors (no external model needed)
    try:
        vec   = TfidfVectorizer(min_df=1, stop_words='english', max_features=500)
        tfidf = vec.fit_transform(texts)
        sim   = cosine_similarity(tfidf)  # n×n attention matrix
    except Exception:
        return [1.0]*len(texts)  # fallback

    # Importance = row sum excluding self-similarity
    np.fill_diagonal(sim, 0)
    raw_importance = sim.sum(axis=1)  # shape (n,)

    # Normalise to [0, 1]
    mn, mx = raw_importance.min(), raw_importance.max()
    if mx == mn:
        return [0.5]*len(msgs)
    return list((raw_importance - mn) / (mx - mn))


def method_tsgc_at(msgs, z1=5, z2=15):
    """TSGC with Attention-based scoring — Transformer-inspired."""

    # Layer 1: deduplication
    seen, deduped = set(), []
    for m in msgs:
        sents  = split_sentences(m['content'])
        unique = [s for s in sents if len(s.strip())<20 or s.strip().lower() not in seen]
        for s in unique: seen.add(s.strip().lower())
        c = ' '.join(unique).strip()
        if c: deduped.append({'role':m['role'],'content':c})

    # Compute attention importance scores for all messages
    importance = message_importance(deduped)  # [0,1] per message

    # Layer 2-AT: attention-gated retention
    result, n = [], len(deduped)
    for i, m in enumerate(deduped):
        ri   = n-1-i
        imp  = importance[i]

        # Base zone (same as TSGC) — positional prior
        if   ri < z1:  zone, base_ratio = 1, 1.0
        elif ri < z2:  zone, base_ratio = 2, 0.40
        else:          zone, base_ratio = 3, 0.15

        # ── Attention gate ───────────────────────────────────────
        # High-importance messages get their ratio boosted
        # regardless of positional zone
        #
        # attention_ratio: maps importance [0,1] → retention [0.15, 1.0]
        attention_ratio = 0.15 + imp * 0.85

        # Final ratio = max(positional, attention)
        # → a pivot message in Zone 3 can be protected up to 100%
        # → a low-importance message in Zone 2 can be compressed harder
        final_ratio = max(base_ratio, attention_ratio) if imp > 0.6 else \
                      min(base_ratio, attention_ratio + 0.1)
        final_ratio = min(1.0, max(0.1, final_ratio))

        c = extract_top(m['content'], final_ratio, m['role']) \
            if final_ratio < 0.95 and len(m['content']) > 80 else m['content']

        result.append({
            'role':m['role'],'content':c,'_zone':zone,
            '_importance':round(imp,2),'_ratio':round(final_ratio,2)
        })
    return result

print('TSGC-AT ready')

## 4. Run Full Benchmark

In [ ]:
METHODS = [
    ('RAW',          lambda m: method_raw(m),         False),
    ('TRUNCATE',     lambda m: method_truncate(m,8),  False),
    ('UNIFORM-50%',  lambda m: method_uniform(m,0.5), False),
    ('DEFLATE ONLY', lambda m: method_deflate_only(m),True),
    ('TSGC',         lambda m: method_tsgc(m),        True),
    ('TSGC-AG',      lambda m: method_tsgc_ag(m),     True),
    ('TSGC-AT',      lambda m: method_tsgc_at(m),     True),
]

all_results = []
for conv_name, conv in CONVERSATIONS.items():
    print(f'\n── {conv_name} ({len(conv)} msgs) ──')
    for name, fn, use_deflate in METHODS:
        compressed = fn(conv)
        res = evaluate(name, conv, compressed, deflate=use_deflate)
        res['Conv'] = conv_name
        res['Type'] = 'NonLinear' if 'NonLinear' in conv_name else 'Linear'
        all_results.append(res)
        print(f'  {name:14} | Save:{res["Savings %"]:5.1f}% '
              f'| KT:{res["Key Term %"]:5.1f}% '
              f'| Pivot:{res["Pivot Recall %"]:5.1f}%')

df = pd.DataFrame(all_results)

METHOD_ORDER = ['RAW','TRUNCATE','UNIFORM-50%','DEFLATE ONLY','TSGC','TSGC-AG','TSGC-AT']
df['Method'] = pd.Categorical(df['Method'], categories=METHOD_ORDER, ordered=True)
df = df.sort_values(['Conv','Method'])

agg = df.groupby('Method')[['Savings %','Key Term %','Recency %','Pivot Recall %','Info Density %']].mean().round(1)
agg_linear    = df[df['Type']=='Linear'].groupby('Method')[['Savings %','Key Term %','Recency %','Pivot Recall %']].mean().round(1)
agg_nonlinear = df[df['Type']=='NonLinear'].groupby('Method')[['Savings %','Key Term %','Recency %','Pivot Recall %']].mean().round(1)

print('\n── Overall Average ──')
print(agg.to_string())

## 5. Graphs

In [ ]:
COLORS = {
    'RAW':MUTED,'TRUNCATE':DANGER,'UNIFORM-50%':WARNING,
    'DEFLATE ONLY':BLUE,'TSGC':ACCENT,'TSGC-AG':TEAL,'TSGC-AT':PINK
}

# ── FIGURE 1: Overall multi-metric bar chart ───────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 6))
fig.suptitle('All Methods — Overall Average Performance', fontsize=15, fontweight='bold', y=1.02)

metrics = [
    ('Savings %',       'Storage Savings (%)'),
    ('Key Term %',      'Key Term Retention (%)'),
    ('Recency %',       'Recency Fidelity (%)'),
    ('Pivot Recall %',  'Pivot Message Recall (%)'),
]

methods_ordered = [m for m in METHOD_ORDER if m in agg.index]
cols = [COLORS[m] for m in methods_ordered]

for ax, (col, title) in zip(axes, metrics):
    vals = [agg.loc[m, col] for m in methods_ordered]
    bars = ax.bar(methods_ordered, vals, color=cols, width=0.6, zorder=3)
    ax.set_title(title, fontsize=11, pad=8)
    ax.set_ylim(0,120); ax.tick_params(axis='x', rotation=30)
    ax.grid(axis='y', zorder=0)
    ax.spines[['top','right','left','bottom']].set_visible(False)
    for bar,v in zip(bars,vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1.5,
                f'{v}%', ha='center', va='bottom', fontsize=8, fontweight='bold', color='#f0f2f8')
    # Highlight TSGC family
    for bar,m in zip(bars, methods_ordered):
        if m in ('TSGC','TSGC-AG','TSGC-AT'):
            bar.set_edgecolor('white'); bar.set_linewidth(1.2)

plt.tight_layout()
plt.savefig('fig1_overall.png', dpi=150, bbox_inches='tight', facecolor='#0d0f14')
plt.show(); print('Saved: fig1_overall.png')

In [ ]:
# ── FIGURE 2: Linear vs Non-Linear — Key hypothesis test ───────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Hypothesis Test: Linear vs Non-Linear Conversations\n'
             'TSGC-AT should gain most on Non-Linear (Pivot Recall)', fontsize=13, fontweight='bold')

tsgc_methods = ['TSGC','TSGC-AG','TSGC-AT']
tsgc_cols    = [ACCENT, TEAL, PINK]

for ax, (agg_data, label) in zip(axes, [(agg_linear,'Linear Conversations'),
                                         (agg_nonlinear,'Non-Linear Conversations')]):
    x     = np.arange(len(tsgc_methods))
    width = 0.22

    metric_labels = ['Savings %','Key Term %','Recency %','Pivot Recall %']
    metric_cols   = [SUCCESS, ACCENT, WARNING, PINK]

    for j,(met,mc) in enumerate(zip(metric_labels, metric_cols)):
        vals = [agg_data.loc[m,met] if m in agg_data.index else 0 for m in tsgc_methods]
        bars = ax.bar(x + j*width - 1.5*width, vals, width, label=met,
                      color=mc, alpha=0.85, zorder=3)
        for bar,v in zip(bars,vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.8,
                    f'{v:.0f}', ha='center', va='bottom', fontsize=7, color='#f0f2f8')

    ax.set_xticks(x); ax.set_xticklabels(tsgc_methods, fontsize=11, fontweight='bold')
    ax.set_ylim(0,120); ax.set_title(label, fontsize=12, pad=10)
    ax.grid(axis='y', zorder=0)
    ax.spines[['top','right','left','bottom']].set_visible(False)
    ax.legend(fontsize=9, facecolor='#1e2330', edgecolor='#2a2d3a', labelcolor='#f0f2f8', loc='upper right')

plt.tight_layout()
plt.savefig('fig2_linear_vs_nonlinear.png', dpi=150, bbox_inches='tight', facecolor='#0d0f14')
plt.show(); print('Saved: fig2_linear_vs_nonlinear.png')

In [ ]:
# ── FIGURE 3: Attention heatmap for non-linear conversation ─────
conv_nl = CONVERSATIONS[list(CONVERSATIONS.keys())[0]]
texts   = [f'M{i+1}: '+m['content'][:60]+'...' for i,m in enumerate(conv_nl)]

vec   = TfidfVectorizer(min_df=1, stop_words='english', max_features=500)
tfidf = vec.fit_transform([m['content'] for m in conv_nl])
sim   = cosine_similarity(tfidf)

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(sim, annot=True, fmt='.2f', cmap='magma',
            xticklabels=[f'M{i+1}' for i in range(len(conv_nl))],
            yticklabels=[f'M{i+1}: {m["content"][:35]}...' for i,m in enumerate(conv_nl)],
            ax=ax, linewidths=0.3, linecolor='#0d0f14',
            cbar_kws={'label':'Cosine Similarity'})

ax.set_title('TSGC-AT: Cross-Message Attention Matrix (Non-Linear Conversation)\n'
             'Bright rows = pivot messages that many others attend to', fontsize=12, pad=15)
ax.tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.savefig('fig3_attention_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#0d0f14')
plt.show(); print('Saved: fig3_attention_heatmap.png')

In [ ]:
# ── FIGURE 4: Tradeoff scatter — Savings vs Pivot Recall ────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('The Compression Tradeoff: Storage Savings vs Semantic Fidelity', fontsize=13, fontweight='bold')

for ax, (agg_d, title) in zip([ax1,ax2],
    [(agg_linear,'Linear Conversations'), (agg_nonlinear,'Non-Linear Conversations')]):

    for m in methods_ordered:
        if m not in agg_d.index: continue
        row = agg_d.loc[m]
        sz  = 400 if m in ('TSGC','TSGC-AG','TSGC-AT') else 150
        ec  = 'white' if m in ('TSGC','TSGC-AG','TSGC-AT') else 'none'
        ax.scatter(row['Savings %'], row['Pivot Recall %'],
                   s=sz, color=COLORS[m], zorder=5, edgecolors=ec, linewidths=1.2)
        offset = 2
        ax.annotate(m, (row['Savings %']+offset, row['Pivot Recall %']+offset),
                    fontsize=9, color=COLORS[m],
                    fontweight='bold' if m in ('TSGC','TSGC-AG','TSGC-AT') else 'normal')

    ax.axhspan(75,115,alpha=0.04,color=SUCCESS)
    ax.axvspan(65,115,alpha=0.04,color=SUCCESS)
    ax.text(66, 76, '★ Ideal zone', fontsize=9, color=SUCCESS, alpha=0.6)
    ax.set_xlabel('Storage Savings (%)')
    ax.set_ylabel('Pivot Message Recall (%)')
    ax.set_title(title, fontsize=11)
    ax.set_xlim(-5,110); ax.set_ylim(20,110)
    ax.grid(zorder=0)
    ax.spines[['top','right']].set_visible(False)

legend_patches = [mpatches.Patch(color=COLORS[m], label=m) for m in methods_ordered]
fig.legend(handles=legend_patches, loc='lower center', ncol=7, fontsize=9,
           facecolor='#1e2330', edgecolor='#2a2d3a', labelcolor='#f0f2f8', bbox_to_anchor=(0.5,-0.05))
plt.tight_layout()
plt.savefig('fig4_tradeoff.png', dpi=150, bbox_inches='tight', facecolor='#0d0f14')
plt.show(); print('Saved: fig4_tradeoff.png')

In [ ]:
# ── FIGURE 5: Radar — TSGC family only ─────────────────────────
cats   = ['Savings','Key Terms','Recency','Pivot Recall','Info Density']
cols_r = ['Savings %','Key Term %','Recency %','Pivot Recall %','Info Density %']
N      = len(cats)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist() + [0]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
fig.patch.set_facecolor('#0d0f14')
ax.set_facecolor('#161921')

for method, color, lw in [('TSGC',ACCENT,2),('TSGC-AG',TEAL,2.5),('TSGC-AT',PINK,3)]:
    vals = [agg.loc[method, c] for c in cols_r] + [agg.loc[method, cols_r[0]]]
    ax.plot(angles, vals, 'o-', lw=lw, color=color, label=method)
    ax.fill(angles, vals, alpha=0.12, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(cats, size=11, color='#f0f2f8')
ax.set_ylim(0, 100)
ax.set_yticks([25,50,75,100])
ax.set_yticklabels(['25','50','75','100'], color=MUTED, size=8)
ax.grid(color='#2a2d3a', linestyle='--')
ax.spines['polar'].set_color('#2a2d3a')
ax.legend(loc='upper right', bbox_to_anchor=(1.35,1.15),
          facecolor='#1e2330', edgecolor='#2a2d3a', labelcolor='#f0f2f8', fontsize=11)
ax.set_title('TSGC Family — Radar Comparison (Overall)', color='#f0f2f8', size=13, pad=20, fontweight='bold')

plt.tight_layout()
plt.savefig('fig5_radar.png', dpi=150, bbox_inches='tight', facecolor='#0d0f14')
plt.show(); print('Saved: fig5_radar.png')

## 6. Summary Table + References

In [ ]:
print('═'*80)
print('TSGC BENCHMARK — Full Summary')
print('═'*80)
print(f'{"Method":<16}{"Savings":>9}{"KeyTerms":>10}{"Recency":>9}{"Pivot":>8}{"Density":>9}')
print('─'*80)
for m in methods_ordered:
    if m not in agg.index: continue
    r = agg.loc[m]
    tag = ' ◄' if m in ('TSGC','TSGC-AG','TSGC-AT') else ''
    print(f"{m:<16}{r['Savings %']:>8.1f}%{r['Key Term %']:>9.1f}%"
          f"{r['Recency %']:>8.1f}%{r['Pivot Recall %']:>7.1f}%{r['Info Density %']:>8.1f}%{tag}")

print('═'*80)
print('\nNON-LINEAR ONLY (Conv: Massive-NonLinear-Callbacks)')
print('─'*80)
for m in ['TSGC','TSGC-AG','TSGC-AT']:
    if m not in agg_nonlinear.index: continue
    r = agg_nonlinear.loc[m]
    print(f"{m:<16} Pivot Recall: {r['Pivot Recall %']:>5.1f}%  Key Terms: {r['Key Term %']:>5.1f}%")

print('\n'+'═'*80)
print('REFERENCES')
print('─'*80)
for r in [
    '[1] Deutsch, P. (1996). DEFLATE Compressed Data Format Specification. RFC 1951.',
    '[2] Mihalcea & Tarau (2004). TextRank: Bringing Order into Texts. EMNLP 2004.',
    '[3] Jiang et al. (2023). LLMLingua: Compressing Prompts for Accelerated Inference. arXiv:2310.05736.',
    '[4] Vaswani et al. (2017). Attention Is All You Need. NeurIPS 2017.',
    '[5] Hochreiter & Schmidhuber (1997). Long Short-Term Memory. Neural Computation.',
    '[6] Reyna & Brainerd (1995). Fuzzy-trace theory. Learning and Individual Differences.',
    '[7] Salton & McGill (1983). Introduction to Modern Information Retrieval. (TF-IDF)',
]: print(r)